# Task
Fine-tune a chosen LLM (Qwen2.5-1.5B, Mistral-7B, Phi-3-mini, or other) using LoRA/QLoRA with either Unsloth or Hugging Face PEFT on the dataset located at "/content/cleaned_data/lawbot_cleaned.jsonl" and evaluate the performance using an appropriate metric.

## Setup environment

### Subtask:
Install necessary libraries like `transformers`, `peft`, `accelerate`, and potentially `unsloth` if chosen.


**Reasoning**:
Install the required libraries for fine-tuning and evaluation.



In [5]:
%pip install transformers peft accelerate
# If planning to use Unsloth, uncomment and run the following based on your CUDA version
# %pip install "unsloth[colab-new] @ git+https://github.com/unsloth/unsloth.git"

## Load data

### Subtask:
Load the `/content/cleaned_data/lawbot_cleaned.jsonl` dataset using the `datasets` library.


In [10]:
%pip install numpy<2

/bin/bash: line 1: 2: No such file or directory


**Reasoning**:
The previous command failed because of a shell issue with the redirect or comparison operator. I need to reinstall numpy specifying a version less than 2 using standard pip syntax.



In [1]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/cleaned_data/lawbot_cleaned.jsonl")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'source'],
        num_rows: 14460
    })
})


## Prepare data

### Subtask:
Format the data for fine-tuning, which may involve tokenization and structuring it into prompts/responses suitable for the chosen model.


**Reasoning**:
Define the function to format the data into a prompt-response structure.



In [2]:
def format_lawbot_data(example):
    # Assuming each example has 'instruction' and 'response' keys
    # You might need to adjust this based on the actual structure of your JSONL
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['response']
    return {"prompt": prompt, "response": response}

**Reasoning**:
Apply the formatting function to the dataset and tokenize the formatted data.



In [3]:
from transformers import AutoTokenizer

# Assuming you want to use a tokenizer compatible with Qwen2-1.5B as an example
# You should choose a tokenizer based on your chosen LLM
model_name = "Qwen/Qwen2-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Set pad token

# Apply the formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Tokenize the dataset
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "response", "prompt"])

print(tokenized_dataset)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

KeyError: 'response'

**Reasoning**:
The error indicates that the key 'response' does not exist in the dataset samples. I need to inspect the dataset structure to identify the correct key names and adjust the formatting function accordingly.



In [4]:
print(dataset['train'][0])

{'instruction': 'What is India according to the Union and its Territory?', 'output': 'India, that is Bharat, shall be a Union of States.', 'source': ['Unknown']}


**Reasoning**:
The dataset sample shows that the response is stored under the key 'output', not 'response'. I will correct the formatting function to use 'output' and re-run the formatting and tokenization steps.



In [5]:
def format_lawbot_data(example):
    # Corrected to use 'output' key
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['output']
    return {"prompt": prompt, "response": response}

# Apply the corrected formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Tokenize the dataset
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "output", "source", "prompt", "response"])

print(tokenized_dataset)

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 14460
    })
})


## Choose and load model

### Subtask:
Select a pre-trained LLM (e.g., Qwen2.5-1.5B, Mistral-7B, Phi-3-mini) and load it using the `transformers` library, potentially in 4-bit quantization for QLoRA.


**Reasoning**:
Load the chosen pre-trained model (Phi-3-mini) and its tokenizer, setting up for potential 4-bit quantization, and ensure the tokenizer has a padding token.



In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "microsoft/Phi-3-mini-4k-instruct" # Chosen LLM

# Configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Apply 4-bit quantization
    trust_remote_code=True,
    torch_dtype=torch.bfloat16, # Use bfloat16 for computation
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.resize_token_embeddings(len(tokenizer))

print("Model and Tokenizer loaded successfully.")
print(f"Model: {model_name}")
print(f"Tokenizer pad token: {tokenizer.pad_token}")

NameError: name 'torch' is not defined

**Reasoning**:
The previous code failed because the `torch` library was not imported before being used in the `BitsAndBytesConfig`. The corrected code block will import torch and then proceed to load the model and tokenizer with the quantization configuration.



In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct" # Chosen LLM

# Configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Apply 4-bit quantization
    trust_remote_code=True,
    torch_dtype=torch.bfloat16, # Use bfloat16 for computation
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.resize_token_embeddings(len(tokenizer))

print("Model and Tokenizer loaded successfully.")
print(f"Model: {model_name}")
print(f"Tokenizer pad token: {tokenizer.pad_token}")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully.
Model: microsoft/Phi-3-mini-4k-instruct
Tokenizer pad token: <|endoftext|>


## Configure lora/qlora

### Subtask:
Set up the LoRA/QLoRA configuration using the `peft` library, specifying parameters like the rank (`r`), alpha (`lora_alpha`), and dropout (`lora_dropout`).


**Reasoning**:
Import the LoraConfig class and define the LoRA configuration object as requested in the instructions.



In [8]:
from peft import LoraConfig

# Define the LoRA configuration
lora_config = LoraConfig(
    r=16,  # LoRA attention dimension
    lora_alpha=16,  # Alpha parameter for LoRA scaling
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type
    task_type="CAUSAL_LM",  # Task type for causal language modeling
)

# Print the LoRA configuration
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules=None, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None)


## Fine-tune model

### Subtask:
Apply the LoRA/QLoRA adapters to the model and train it on the prepared dataset using the `transformers` Trainer or a custom training loop.


**Reasoning**:
Apply the LoRA configuration to the model, define training arguments, instantiate the Trainer, and start the training process as per the instructions.



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
import os

# Define the new model name
model_name = "mistralai/Mistral-7B-v0.1"

# Configuration for 4-bit quantization (keeping the same config as before for now)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the new model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.resize_token_embeddings(len(tokenizer))

# Define the target modules for Mistral-7B
# Common target modules for Mistral include 'q_proj', 'k_proj', 'v_proj', 'o_proj'
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# Define the LoRA configuration with target modules for Mistral
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Wrap the base model with PeftModel
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")


# Print the trainable parameters
peft_model.print_trainable_parameters()

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./mistral_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    report_to="none",
)

# Disable caching explicitly in the model config
peft_model.config.use_cache = False

# Define data collator with pad_to_multiple_of=8
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

# Instantiate Trainer with the PEFT model
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'], # Use the tokenized dataset from previous step
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

**Reasoning**:
The previous attempt to train Mistral-7B with the same quantization config resulted in a CUDA error. I will now try adjusting the `BitsAndBytesConfig` parameters, specifically trying a different `bnb_4bit_quant_type` and potentially disabling `bnb_4bit_use_double_quant` if it's still enabled (though it was disabled in the last attempt), to see if this resolves the CUDA error. I will also ensure caching is explicitly disabled.



In [1]:
# Re-configure BitsAndBytesConfig with different parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="fp4", # Try 'fp4' quant type instead of 'nf4'
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False, # Ensure double quantization is off
)

# Load the model again with the updated quantization config
# This will replace the previously loaded model
model = AutoModelForCausalLM.from_pretrained(
    model_name, # Use the previously defined model_name (Mistral-7B)
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Re-wrap the base model with PeftModel using the same lora_config
# Check if the model is already a PeftModel before wrapping again
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config) # Use the previously defined lora_config
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args, # Use the previously defined training_args
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

NameError: name 'BitsAndBytesConfig' is not defined

**Reasoning**:
The previous code block failed because `BitsAndBytesConfig` was not defined. I need to ensure all necessary classes are imported before they are used. I will regenerate the full code block, including the necessary imports from `transformers` and `peft`, to re-attempt loading the model with the adjusted quantization configuration and start training.



In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
import os

# Re-configure BitsAndBytesConfig with different parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="fp4", # Try 'fp4' quant type instead of 'nf4'
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False, # Ensure double quantization is off
)

# Load the model again with the updated quantization config
# This will replace the previously loaded model
model = AutoModelForCausalLM.from_pretrained(
    model_name, # Use the previously defined model_name (Mistral-7B)
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Re-wrap the base model with PeftModel using the same lora_config
# Check if the model is already a PeftModel before wrapping again
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config) # Use the previously defined lora_config
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args, # Use the previously defined training_args
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

NameError: name 'model_name' is not defined

**Reasoning**:
The previous code block failed because `model_name` was not defined. I need to ensure all variables used in the code block, including `model_name`, `lora_config`, `training_args`, `tokenized_dataset`, `tokenizer`, and `data_collator`, are defined within or accessible by the block. I will regenerate the full code block, including the definitions of these variables (reusing the values from previous successful steps), to re-attempt loading the model with the adjusted quantization configuration and start training.



In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
import os

# Define the model name (reusing from previous step)
model_name = "mistralai/Mistral-7B-v0.1"

# Define the target modules for Mistral-7B (reusing from previous step)
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Define the LoRA configuration (reusing from previous step)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Define TrainingArguments (reusing from previous step)
training_args = TrainingArguments(
    output_dir="./mistral_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    report_to="none",
)

# Re-configure BitsAndBytesConfig with different parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="fp4", # Try 'fp4' quant type instead of 'nf4'
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False, # Ensure double quantization is off
)

# Load the model again with the updated quantization config
# This will replace the previously loaded model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Assuming tokenizer, tokenized_dataset, and data_collator are still available from previous successful steps.
# If not, they would need to be re-defined here as well.
# For this attempt, assuming they persist in the environment.

# Re-wrap the base model with PeftModel using the same lora_config
# Check if the model is already a PeftModel before wrapping again
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


NameError: name 'tokenized_dataset' is not defined

**Reasoning**:
The previous code block failed because `tokenized_dataset`, `tokenizer`, and `data_collator` were not defined. These variables were created in earlier successful steps but need to be accessible in the current block. I will regenerate the full code block, including the definitions of `tokenized_dataset`, `tokenizer`, and `data_collator` (re-executing the data preparation steps), to ensure all necessary variables are available for the Trainer.



In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import os

# Re-load the dataset
dataset = load_dataset("json", data_files="/content/cleaned_data/lawbot_cleaned.jsonl")

# Define the data formatting function (reusing from previous step)
def format_lawbot_data(example):
    # Corrected to use 'output' key
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['output']
    return {"prompt": prompt, "response": response}

# Apply the corrected formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Define the model name (reusing from previous step)
model_name = "mistralai/Mistral-7B-v0.1"

# Re-load the tokenizer (reusing from previous step)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Note: Resizing embeddings might not be necessary if tokenizer is re-loaded
    # model.resize_token_embeddings(len(tokenizer))

# Define the tokenization function (reusing from previous step)
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

# Re-tokenize the dataset
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "output", "source", "prompt", "response"])

# Define the data collator with pad_to_multiple_of=8 (reusing from previous step)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)


# Define the target modules for Mistral-7B (reusing from previous step)
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Define the LoRA configuration (reusing from previous step)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Define TrainingArguments (reusing from previous step)
training_args = TrainingArguments(
    output_dir="./mistral_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    report_to="none",
)

# Re-configure BitsAndBytesConfig with different parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="fp4", # Try 'fp4' quant type instead of 'nf4'
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False, # Ensure double quantization is off
)

# Load the model again with the updated quantization config
# This will replace the previously loaded model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Re-wrap the base model with PeftModel using the same lora_config
# Check if the model is already a PeftModel before wrapping again
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-2975511536.py:106: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 70.12 MiB is free. Process 135117 has 14.67 GiB memory in use. Of the allocated memory 14.43 GiB is allocated by PyTorch, and 115.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

**Reasoning**:
The training failed with an `OutOfMemoryError`, indicating that even with 4-bit quantization and the Mistral-7B model, the GPU memory is insufficient. The traceback points to the matrix multiplication operation within the 4-bit linear layer. To address this, I will try reducing the `per_device_train_batch_size` and increasing `gradient_accumulation_steps` to effectively reduce the memory used per step while keeping the effective batch size the same. I will also try a smaller model, Phi-3-mini, but train it in full precision if memory allows, or adjust the quantization parameters further. Given the memory constraints, trying a smaller model with potentially less aggressive quantization or full precision is a reasonable next step. I will revert to Phi-3-mini as it's smaller, and attempt full precision training first. If that fails, I will try different quantization settings on Phi-3-mini again.



In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import os

# Re-load the dataset
dataset = load_dataset("json", data_files="/content/cleaned_data/lawbot_cleaned.jsonl")

# Define the data formatting function (reusing from previous step)
def format_lawbot_data(example):
    # Corrected to use 'output' key
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['output']
    return {"prompt": prompt, "response": response}

# Apply the corrected formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Define the model name (reverting to Phi-3-mini as it's smaller)
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Re-load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Note: Resizing embeddings might not be necessary if tokenizer is re-loaded
    # model.resize_token_embeddings(len(tokenizer))

# Define the tokenization function (reusing from previous step)
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

# Re-tokenize the dataset
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "output", "source", "prompt", "response"])

# Define the data collator with pad_to_multiple_of=8 (reusing from previous step)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)


# Define the target modules for Phi-3-mini (reusing from previous step)
target_modules = ["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# Define the LoRA configuration (reusing from previous step)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Define TrainingArguments with reduced batch size and increased gradient accumulation
training_args = TrainingArguments(
    output_dir="./phi3_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduced batch size
    gradient_accumulation_steps=8,  # Increased accumulation steps (effective batch size remains 16)
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,  # Keep mixed precision for now
    push_to_hub=False,
    report_to="none",
)

# Load the model without quantization for full precision training attempt
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16, # Use bfloat16 for full precision if possible
    device_map="auto",
)


# Wrap the base model with PeftModel using the same lora_config
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-942986385.py:97: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 15,204,352 || all params: 3,836,283,904 || trainable%: 0.3963


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 135117 has 14.73 GiB memory in use. Of the allocated memory 14.48 GiB is allocated by PyTorch, and 122.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

**Reasoning**:
Full precision training of Phi-3-mini also resulted in an `OutOfMemoryError`. This suggests that even the smaller model in full precision with the current batch size and gradient accumulation is too large for the available memory. To address this, I will revert to 4-bit quantization, but try different `BitsAndBytesConfig` settings on Phi-3-mini. Specifically, I will try `bnb_4bit_quant_type="nf4"` again (as it's generally recommended) but ensure `bnb_4bit_use_double_quant` is set to `True` this time, as double quantization can further reduce memory usage. I will keep the reduced `per_device_train_batch_size` and increased `gradient_accumulation_steps`.



In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import os

# Re-load the dataset
dataset = load_dataset("json", data_files="/content/cleaned_data/lawbot_cleaned.jsonl")

# Define the data formatting function (reusing from previous step)
def format_lawbot_data(example):
    # Corrected to use 'output' key
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['output']
    return {"prompt": prompt, "response": response}

# Apply the corrected formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Define the model name (staying with Phi-3-mini)
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Re-load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Note: Resizing embeddings might not be necessary if tokenizer is re-loaded
    # model.resize_token_embeddings(len(tokenizer))

# Define the tokenization function (reusing from previous step)
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

# Re-tokenize the dataset
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "output", "source", "prompt", "response"])

# Define the data collator with pad_to_multiple_of=8 (reusing from previous step)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)


# Define the target modules for Phi-3-mini (reusing from previous step)
target_modules = ["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# Define the LoRA configuration (reusing from previous step)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Define TrainingArguments with reduced batch size and increased gradient accumulation (reusing from previous step)
training_args = TrainingArguments(
    output_dir="./phi3_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduced batch size
    gradient_accumulation_steps=8,  # Increased accumulation steps (effective batch size remains 16)
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,  # Keep mixed precision
    push_to_hub=False,
    report_to="none",
)

# Load the model with 4-bit quantization, trying nf4 with double quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Use 'nf4' quant type
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # Enable double quantization
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


# Wrap the base model with PeftModel using the same lora_config
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 15,204,352 || all params: 3,836,283,904 || trainable%: 0.3963


/tmp/ipython-input-2430225055.py:105: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


TypeError: device() received an invalid combination of arguments - got (NoneType), but expected one of:
 * (torch.device device)
      didn't match because some of the arguments have invalid types: (!NoneType!)
 * (str type, int index = -1)


**Reasoning**:
The training failed with a `TypeError` related to `torch.device()`, suggesting an issue with device placement or how the device is being handled by Accelerate or bitsandbytes in this configuration. The traceback indicates the error occurs within Accelerate's `prepare_model` function, likely when trying to determine the current device. While the exact cause is unclear, it could be related to the interaction between `device_map="auto"`, bitsandbytes, and Accelerate. To attempt to fix this, I will explicitly set `device_map` to `None` when loading the model and rely on Accelerate to handle device placement. This might resolve the conflict. I will keep the 4-bit quantization settings with double quantization and the adjusted training parameters.



In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import os

# Re-load the dataset
dataset = load_dataset("json", data_files="/content/cleaned_data/lawbot_cleaned.jsonl")

# Define the data formatting function (reusing from previous step)
def format_lawbot_data(example):
    # Corrected to use 'output' key
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    response = example['output']
    return {"prompt": prompt, "response": response}

# Apply the corrected formatting function
formatted_dataset = dataset.map(format_lawbot_data)

# Define the model name (staying with Phi-3-mini)
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Re-load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Note: Resizing embeddings might not be necessary if tokenizer is re-loaded
    # model.resize_token_embeddings(len(tokenizer))

# Define the tokenization function (reusing from previous step)
def tokenize_function(examples):
    # Concatenate prompt and response for training
    text = [f"{p} {r}{tokenizer.eos_token}" for p, r in zip(examples["prompt"], examples["response"])]
    return tokenizer(text, padding="max_length", truncation=True, max_length=512) # Adjust max_length as needed

# Re-tokenize the dataset
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "output", "source", "prompt", "response"])

# Define the data collator with pad_to_multiple_of=8 (reusing from previous step)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)


# Define the target modules for Phi-3-mini (reusing from previous step)
target_modules = ["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# Define the LoRA configuration (reusing from previous step)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# Define TrainingArguments with reduced batch size and increased gradient accumulation (reusing from previous step)
training_args = TrainingArguments(
    output_dir="./phi3_lawbot_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduced batch size
    gradient_accumulation_steps=8,  # Increased accumulation steps (effective batch size remains 16)
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,  # Keep mixed precision
    push_to_hub=False,
    report_to="none",
)

# Load the model with 4-bit quantization, nf4 with double quantization, and device_map=None
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Use 'nf4' quant type
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # Enable double quantization
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=None, # Set device_map to None
)


# Wrap the base model with PeftModel using the same lora_config
if not isinstance(model, PeftModel):
    peft_model = get_peft_model(model, lora_config)
else:
    peft_model = model
    print("Model is already a PEFT model. Skipping get_peft_model.")

# Print the trainable parameters
peft_model.print_trainable_parameters()

# Ensure caching is explicitly disabled in the model config
peft_model.config.use_cache = False

# Instantiate Trainer with the new PEFT model (using previously defined args, dataset, tokenizer, and data_collator)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Map:   0%|          | 0/14460 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 15,204,352 || all params: 3,836,283,904 || trainable%: 0.3963


/tmp/ipython-input-3065664031.py:105: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,2.145100
20,1.666800
30,1.617100
40,1.522100
50,1.495000
60,1.453300
70,1.487000
80,1.442600
90,1.466900
100,1.484800


KeyboardInterrupt: 